In [ ]:
pip install torch

In [ ]:

import torch
from torch import linalg as LA
import os
import time
import statistics
from typing import Tuple
from typing_extensions import Annotated

In [ ]:
import os, platform, re, subprocess, sys
import psutil

def cpu_name():
    if sys.platform == "darwin":
        return subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                              capture_output=True, text=True).stdout.strip()
    if sys.platform.startswith("linux"):
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    if sys.platform == "win32":
        import winreg
        with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,
                            r"HARDWARE\DESCRIPTION\System\CentralProcessor\0") as k:
            return winreg.QueryValueEx(k, "ProcessorNameString")[0].strip()
    return platform.processor()

print("chip:", cpu_name())
print("physical cores:", psutil.cpu_count(logical=False))
print("logical cores:", psutil.cpu_count(logical=True))
print("memory (GB):", psutil.virtual_memory().total / 2**30)
print("OS:", platform.platform(), platform.machine())
print("python:", sys.executable)
print("torch:", torch.__version__, re.findall(r"BLAS_INFO=\w+", torch.__config__.show()))

In [ ]:
torch.manual_seed(42)
a= torch.randn(500000000,device="cpu")


In [ ]:
def exp(a:torch.tensor)->torch.tensor:
    expo=torch.exp(a)
    return expo


In [ ]:
def time_counter(no_of_threads:int,a:torch.tensor,results:list):
    n_dict={}
    n_lst=[]

    torch.set_num_threads(no_of_threads)
    print(torch.get_num_threads())
    time.sleep(2)
    exp(a)
    for i in range(10):
        
        start_time=time.perf_counter()
        exp(a)
        end_time=time.perf_counter()
        n_lst.append(end_time-start_time)
    mean=statistics.mean(n_lst)
    stndev=statistics.stdev(n_lst)
    n_dict['thread']=torch.get_num_threads()
    n_dict['mean']=mean
    n_dict['stdev']=stndev
  
    results.append(n_dict)
    

    

In [ ]:
import pandas as pd
from datetime import datetime
run_name = datetime.now().strftime("mkl_pilot1_exp_%Y-%m-%d_%H-%M")
os.makedirs("results", exist_ok=True)
csv_file = f"results/{run_name}.csv"
print("saving to", csv_file)

results=[]
for i in range(1, 17):
    time_counter(i,a,results)
    pd.DataFrame(results).to_csv(csv_file, index=False)


In [ ]:
results

In [ ]:
print(torch.__config__.show())

In [ ]:
n_results = results   # measured on this laptop by the loop above

t1=n_results[0]['mean']
n_res=[]
for i in range(len(n_results)):
    n_s_e={}
    speedup=t1/n_results[i]['mean']
    efficiency=(speedup/n_results[i]['thread'])*100
    n_s_e['threads']=n_results[i]['thread']
    n_s_e['speedup']=speedup
    n_s_e['efficiency']=efficiency
    n_res.append(n_s_e)





In [ ]:
n_res

In [ ]:
pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

RUN_NAME = f"results/{run_name}"   # plots are saved in results/ with this run's name

threads = [r["threads"] for r in n_res]
speedup = [r["speedup"] for r in n_res]
efficiency = [r["efficiency"] for r in n_res]
# Amdahl serial fraction (%), skipping 1 thread
f_threads = [n for n in threads if n > 1]
serial = [((n / s) - 1) / (n - 1) * 100 for n, s in zip(threads, speedup) if n > 1]

BLUE, GREY = "#2a78d6", "#8a8980"

def tidy(ax, ylabel, title):
    ax.set_xlabel("Threads (N)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(threads)
    ax.grid(True, color="#e4e3dc")
    ax.spines[["top", "right"]].set_visible(False)

# 1. Speedup vs ideal
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, threads, "--", color=GREY, label="Ideal (S = N)")
ax.plot(threads, speedup, "o-", color=BLUE, linewidth=2, label="Measured")
tidy(ax, "Speedup (T1 / TN)", "Speedup vs threads")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_speedup.png", dpi=200); plt.show()

# 2. Efficiency
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, efficiency, "o-", color=BLUE, linewidth=2)
ax.set_ylim(0, 110)
tidy(ax, "Parallel efficiency (%)", "Parallel efficiency vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_efficiency.png", dpi=200); plt.show()

# 3. Serial fraction
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(f_threads, serial, "o-", color=BLUE, linewidth=2)
ax.set_ylim(bottom=0)
tidy(ax, "Implied serial fraction f (%)", "Amdahl serial fraction vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_serial_fraction.png", dpi=200); plt.show()

In [ ]:
# Commit and push this run's results to GitHub.
# The results are written to files in results/ by the code itself, so they are
# committed even if VS Code has not saved the notebook's latest outputs yet.
import glob, os, shutil, subprocess, time
from datetime import datetime
import pandas as pd

NOTEBOOK = "pilot_run1_x86.ipynb"

summary_file = f"results/{run_name}_summary.csv"
pd.DataFrame(n_res).to_csv(summary_file, index=False)

time.sleep(5)   # give VS Code auto save time to write the latest outputs into the notebook

FILES = [NOTEBOOK] + sorted(glob.glob(f"results/{run_name}*"))
print("committing:", *FILES, sep="\n  ")
MESSAGE = f"x86 MKL pilot 1 (exp), threads 1-16 ({datetime.now():%Y-%m-%d %H:%M})"

def git(*args, check=True):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError("git " + args[0] + " failed")
    return r.returncode

if shutil.which("git") is None:
    raise RuntimeError("git is not installed or not on PATH")

git("add", "--", *FILES)
if git("diff", "--cached", "--quiet", check=False) == 0:
    print("nothing new to commit")
else:
    git("commit", "-m", MESSAGE)
git("pull", "--rebase", "--autostash")
git("push")
